# LatentDriver Validation Preprocess (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/latentdriver-waymax-experiments/blob/main/notebooks/latentdriver_preprocess_val_colab.ipynb)

This notebook prepares either a **smoke validation subset** or the **full validation preprocessing artifacts** required for Waymax evaluation, using either a **Drive-local WOMD root** or an **authenticated WOMD GCS root**.


## What this notebook does

1. sync this repo and install the local package
2. mount Google Drive and bind the canonical asset layout
3. clone and patch the pinned LatentDriver fork
4. optionally install the heavy Waymax/JAX/Torch runtime
5. choose whether WOMD validation comes from Drive or authenticated GCS
6. for smoke mode, stage one validation shard locally if needed
7. prepare a one-shard smoke TFRecord if needed
8. run the upstream validation preprocessing job


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/achyutmorang/latentdriver-waymax-experiments.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/latentdriver-waymax-experiments")


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None, stream: bool = True) -> str:
    printable = " ".join(str(part) for part in cmd)
    print(f"$ {printable}")
    if stream:
        proc = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        lines: list[str] = []
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            lines.append(line)
        code = proc.wait()
        output = "".join(lines)
        if code != 0:
            raise RuntimeError(f"Command failed with exit code {code}: {printable}")
        return output
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=True,
    )
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    return completed.stdout


if REPO_DIR.exists():
    run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH])
    run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH])
else:
    run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)])

run([sys.executable, "-m", "pip", "install", "-e", str(REPO_DIR)])
repo_rev = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(json.dumps({"repo_dir": str(REPO_DIR), "repo_rev": repo_rev}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
import sys

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/waymax_research"
os.environ.pop("LATENTDRIVER_RESULTS_ROOT", None)
sys.path.insert(0, str(REPO_DIR / "src"))
from latentdriver_waymax_experiments.colab import bind_drive_layout

binding = bind_drive_layout(DRIVE_ROOT)
print(json.dumps(binding, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json

run([sys.executable, "scripts/bootstrap_upstream.py"], cwd=REPO_DIR)
lock_path = REPO_DIR / "artifacts" / "locks" / "upstream_bootstrap.json"
print(lock_path.read_text())


In [ ]:
INSTALL_RUNTIME = True

if INSTALL_RUNTIME:
    run([sys.executable, "scripts/setup_colab_runtime.py", "--editable-project"], cwd=REPO_DIR)
else:
    print("Skipping runtime installation. Set INSTALL_RUNTIME=True if this is a fresh Colab session.")


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

WOMD_SOURCE_MODE = 'gcs'  # 'drive' or 'gcs'
WAYMO_DATASET_DRIVE_ROOT = '/content/drive/MyDrive/waymo_open_dataset'
WOMD_GCS_DATASET_ROOT = 'gs://waymo_open_dataset_motion_v_1_1_0'
PREPROCESS_MODE = 'smoke'  # switch to 'full' after smoke passes
SMOKE_SHARD_INDEX = 0
GCS_STAGING_ROOT = str(Path(binding["drive_root"]) / "assets" / "raw_womd")

print(json.dumps({
    'WOMD_SOURCE_MODE': WOMD_SOURCE_MODE,
    'WAYMO_DATASET_DRIVE_ROOT': WAYMO_DATASET_DRIVE_ROOT,
    'WOMD_GCS_DATASET_ROOT': WOMD_GCS_DATASET_ROOT,
    'PREPROCESS_MODE': PREPROCESS_MODE,
    'SMOKE_SHARD_INDEX': SMOKE_SHARD_INDEX,
    'GCS_STAGING_ROOT': GCS_STAGING_ROOT,
}, indent=2, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys

if WOMD_SOURCE_MODE not in {'drive', 'gcs'}:
    raise ValueError(f'Unsupported WOMD_SOURCE_MODE={WOMD_SOURCE_MODE!r}')

resolved_dataset_root = WAYMO_DATASET_DRIVE_ROOT
staging_summary = None
gcs_probe_lines: list[str] = []

if WOMD_SOURCE_MODE == 'gcs':
    from google.colab import auth

    auth.authenticate_user()
    adc = subprocess.run(
        ['gcloud', 'auth', 'application-default', 'print-access-token'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    if adc.returncode != 0:
        raise RuntimeError('Failed to acquire application-default credentials for WOMD GCS access.')

    validation_probe = WOMD_GCS_DATASET_ROOT.rstrip("/") + "/uncompressed/tf_example/validation/validation_tfexample.tfrecord-00000-of-00150"
    probe = subprocess.run(
        ["bash", "-lc", f"gsutil ls {shlex.quote(validation_probe)}"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        check=False,
    )
    if probe.returncode != 0:
        detail = probe.stderr.strip() or probe.stdout.strip()
        raise RuntimeError(f'WOMD GCS probe failed for {validation_probe}: {detail}')
    gcs_probe_lines = [line for line in probe.stdout.splitlines() if line.strip()]

    if PREPROCESS_MODE == 'smoke':
        staging_summary = json.loads(run([
            sys.executable,
            'scripts/stage_womd_validation_shard.py',
            '--gcs-root', WOMD_GCS_DATASET_ROOT,
            '--staging-root', GCS_STAGING_ROOT,
            '--shard-index', str(SMOKE_SHARD_INDEX),
        ], cwd=REPO_DIR, stream=False))
        resolved_dataset_root = staging_summary["staging_root"]
    else:
        resolved_dataset_root = WOMD_GCS_DATASET_ROOT

os.environ["LATENTDRIVER_WAYMO_DATASET_ROOT"] = resolved_dataset_root
print(json.dumps({
    'LATENTDRIVER_WAYMO_DATASET_ROOT': os.environ['LATENTDRIVER_WAYMO_DATASET_ROOT'],
    'PREPROCESS_MODE': PREPROCESS_MODE,
    'SMOKE_SHARD_INDEX': SMOKE_SHARD_INDEX,
    'WOMD_SOURCE_MODE': WOMD_SOURCE_MODE,
    'gcs_probe_matches': gcs_probe_lines,
    'staging_summary': staging_summary,
}, indent=2, sort_keys=True))


In [ ]:
if PREPROCESS_MODE == 'smoke':
    run([sys.executable, "scripts/prepare_smoke_subset.py", "--shard-index", str(SMOKE_SHARD_INDEX)], cwd=REPO_DIR)
else:
    print("Skipping smoke subset creation because PREPROCESS_MODE=full")

run([sys.executable, "scripts/preprocess_validation_only.py", "--mode", PREPROCESS_MODE, "--dry-run"], cwd=REPO_DIR)


In [ ]:
RUN_PREPROCESS = True

if RUN_PREPROCESS:
    run([sys.executable, "scripts/preprocess_validation_only.py", "--mode", PREPROCESS_MODE], cwd=REPO_DIR)
else:
    print("Skipping preprocessing run.")


In [ ]:
import json

config = json.loads((REPO_DIR / "configs" / "baselines" / "latentdriver_waymax_eval.json").read_text())
root = REPO_DIR / config["assets"]["preprocessed_root"] / PREPROCESS_MODE
summary = {}
for child in sorted(root.glob("*")):
    summary[child.name] = {
        "path": str(child),
        "exists": child.exists(),
        "entries": len(list(child.glob("*"))) if child.is_dir() else None,
    }
print(json.dumps(summary, indent=2, sort_keys=True))


## Next step

Once preprocessing artifacts exist, run [`latentdriver_public_eval_colab.ipynb`](./latentdriver_public_eval_colab.ipynb) for checkpoint evaluation under a standardized Waymax contract.
